# CAMELS preprocessing v2.2
Versione lineare e compatta della pipeline di preprocessing: dalle particelle alle patch con label.


## 1) Configurazione, percorsi e parametri
Definizione degli input/output e dei parametri fisici/numerici usati in tutta la pipeline.


In [ ]:
import os
import numpy as np
import h5py
import hdf5plugin
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.ndimage import maximum_filter, gaussian_filter
from scipy.stats import spearmanr          # P2-03: correlazione CH1/CH2
import sphviewer2                          # type: ignore

# ── Percorsi file ─────────────────────────────────────────────────────────────
SNAPSHOT_PATH = r"C:\Users\kekko\Downloads\Python\TESI\CAMELS_data\snapshot_090.hdf5"
OUTPUT_DIR    = r"C:\Users\kekko\Downloads\Python\TESI\CAMELS_data\preprocessed_v2.1"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Parametri SPH / immagine (invariati) ─────────────────────────────────────
N_NEIGHBORS_H  = 32
TARGET_CELLS   = 4
R_MAX_LEVEL    = 10
IMG_SIZE       = 2 ** R_MAX_LEVEL    # 1024

# ── Slicing ───────────────────────────────────────────────────────────────────
N_SLICES = 25    # fisso, indicato dal prof. Benitez

# ── Parametri rilevamento picchi ──────────────────────────────────────────────
PEAK_MIN_DISTANCE     = 5      # px: due picchi entro N px vengono uniti
PEAK_THRESHOLD_SIG    = 2.5    # sigma sopra il mediano per accettare un picco
RADIUS_DROP_FACTOR    = np.e   # criterio 1/e: raggio dove ρ scende a peak/e
GAUSSIAN_SIGMA_FACTOR = 0.33   # σ_blob = R_px × factor (default CenterNet)
DEFAULT_RADIUS_PX     = 3.0    # raggio di fallback se stima dal profilo fallisce

# ── Dataset ───────────────────────────────────────────────────────────────────
N_MIN_PARTICLES = 100
PATCH_SIZE      = 256
EPS             = 1e-10    # evita log(0); qui in cella 1 per i test unitari (5b)

print(f"{'IMG_SIZE':<24}: {IMG_SIZE} × {IMG_SIZE} px")
print(f"{'N_SLICES':<24}: {N_SLICES}")
print(f"{'PATCH_SIZE':<24}: {PATCH_SIZE} × {PATCH_SIZE} px")
print(f"{'Patches per immagine':<24}: {(IMG_SIZE // PATCH_SIZE)**2}")
print(f"{'Patch totali (1 sim)':<24}: {N_SLICES * (IMG_SIZE // PATCH_SIZE)**2}")
print(f"{'Output directory':<24}: {OUTPUT_DIR}")


## 2) Caricamento snapshot, geometria delle slice e smoothing lengths
Lettura dello snapshot CAMELS, calcolo della geometria della griglia e caricamento/calcolo degli smoothing lengths.


In [ ]:
# %% ── 2. Caricamento snapshot ───────────────────────────────────────────────

with h5py.File(SNAPSHOT_PATH, "r") as f:
    header       = f["Header"].attrs
    Lbox_ckpc    = float(header["BoxSize"])
    Lbox         = Lbox_ckpc / 1000.0
    redshift     = float(header["Redshift"])
    h_cosmo      = float(header["HubbleParam"])
    omega_m      = float(header["Omega0"])
    mass_table   = np.asarray(header["MassTable"], dtype=np.float64)
    dm_mass_1e10 = mass_table[1]
    dm_mass      = dm_mass_1e10 * 1e10    # M_sun/h

    pos = np.asarray(f["PartType1/Coordinates"][:], dtype=np.float64) / 1000.0
    vel = np.asarray(f["PartType1/Velocities"][:],  dtype=np.float64)

x = np.ascontiguousarray(pos[:, 0])
y = np.ascontiguousarray(pos[:, 1])
z = np.ascontiguousarray(pos[:, 2])
n_particles = len(x)
m = np.full(n_particles, dm_mass, dtype=np.float64)

SLICE_DZ       = Lbox / N_SLICES
pixel_size_mpc = Lbox / IMG_SIZE

print("=== Snapshot summary ===")
print(f"  N particelle  : {n_particles:,}  ({round(n_particles**(1/3))}³)")
print(f"  Box size      : {Lbox:.4f} Mpc/h  ({Lbox_ckpc:.1f} ckpc/h)")
print(f"  Redshift      : {redshift:.6f}")
print(f"  DM mass/part  : {dm_mass:.4e} M☉/h")
print(f"\n=== Slice geometry ===")
print(f"  N_SLICES      : {N_SLICES}")
print(f"  Slice Δz      : {SLICE_DZ:.4f} Mpc/h  ({SLICE_DZ*1000:.1f} ckpc/h)")
print(f"  Pixel size    : {pixel_size_mpc*1000:.2f} ckpc/h")

MASS_CUT_MSUN = N_MIN_PARTICLES * dm_mass
MASS_CUT_CAT  = MASS_CUT_MSUN / 1e10

print("=== Mass cut (riferimento, non usato per le label) ===")
print(f"  MASS_CUT : {MASS_CUT_MSUN:.3e} M☉/h  ({MASS_CUT_CAT:.4f} × 10¹⁰ M☉/h)")
print(f"  r_min risolvibile : {2*pixel_size_mpc*1000:.2f} ckpc/h  (2 pixel)")


# ── Smoothing lengths SPH ───────────────────────────────────────────────────

H_SAVE_PATH = os.path.join(OUTPUT_DIR, f"smoothing_lengths_090_k{N_NEIGHBORS_H}.npy")

if os.path.exists(H_SAVE_PATH):
    print(f"Carico smoothing lengths pre-calcolate:\n  {H_SAVE_PATH}")
    h_sph = np.load(H_SAVE_PATH)
    print(f"  Caricate: {len(h_sph):,} valori")
else:
    print(f"Calcolo smoothing lengths (k={N_NEIGHBORS_H} vicini)...")
    print("  Tempo stimato: 1-3 minuti per 16M particelle.")
    h_sph = sphviewer2.estimate_h(
        x, y, z,
        Lbox=Lbox,
        k=N_NEIGHBORS_H,
        num_threads=os.cpu_count(),
    )
    np.save(H_SAVE_PATH, h_sph)
    print(f"  Salvate in: {H_SAVE_PATH}")

valid_h = np.isfinite(h_sph) & (h_sph > 0)
if not np.all(valid_h):
    n_bad = (~valid_h).sum()
    print(f"  Rimosse {n_bad:,} particelle con h_sph non valide.")
    x, y, z, m, vel = x[valid_h], y[valid_h], z[valid_h], m[valid_h], vel[valid_h]
    h_sph = h_sph[valid_h]

print(f"\nh_sph  min={h_sph.min():.4f}  median={np.median(h_sph):.4f}  "
      f"max={h_sph.max():.4f}  Mpc/h")


## 3) Funzioni di pipeline
Funzioni per rendering/feature map, picchi, stima del raggio, label periodiche e de-duplicazione 3D.


In [ ]:
def project_h_to_grid(x_s, y_s, h_s, lbox, img_size):
    """
    Proietta h_sph su griglia 2D (media per pixel) + smooth periodico.

    """
    ix = np.clip((x_s / lbox * img_size).astype(np.int64), 0, img_size - 1)
    iy = np.clip((y_s / lbox * img_size).astype(np.int64), 0, img_size - 1)

    sum_h  = np.zeros((img_size, img_size), dtype=np.float64)
    count  = np.zeros((img_size, img_size), dtype=np.int32)
    np.add.at(sum_h,  (iy, ix), h_s)
    np.add.at(count,  (iy, ix), 1)

    filled       = count > 0
    h_map        = np.empty((img_size, img_size), dtype=np.float64)
    h_map[filled]  = sum_h[filled] / count[filled]
    h_map[~filled] = h_s.max()    # void → h massimo (bassa densità)

    # P2-02: smooth periodico per ridurre il rumore di binning
    h_map = gaussian_filter(h_map, sigma=1.0, mode='grid-wrap')
    return h_map


def find_density_peaks(log_img, min_distance_px=PEAK_MIN_DISTANCE,
                       threshold_sigma=PEAK_THRESHOLD_SIG, smooth_sigma=1.5):
    """
    Trova massimi locali nell'immagine log10(densità).

    """
    # P0-03: mode='grid-wrap' per box periodico
    smoothed = gaussian_filter(log_img, sigma=smooth_sigma, mode='grid-wrap')

    bg  = np.median(smoothed)
    sig = (np.percentile(smoothed, 84) - np.percentile(smoothed, 16)) / 2.0
    threshold = bg + threshold_sigma * sig

    # P1-02: jitter deterministico per rompere i plateau prima del max_filter
    rng = np.random.default_rng(42)
    smoothed_j = smoothed + rng.uniform(0.0, 1e-6, size=smoothed.shape)

    footprint    = np.ones((min_distance_px * 2 + 1,) * 2, dtype=bool)
    # P0-03: mode='grid-wrap' anche nel maximum_filter
    neighborhood = maximum_filter(smoothed_j, footprint=footprint, mode='grid-wrap')
    peaks_mask   = (smoothed_j == neighborhood) & (smoothed >= threshold)

    peaks_rc  = np.argwhere(peaks_mask)
    peak_vals = smoothed[peaks_mask]
    order = np.argsort(peak_vals)[::-1]
    # P0-02: restituisce smoothed (non grezza) per la stima dei raggi
    return peaks_rc[order], peak_vals[order], smoothed


def estimate_radius_from_profile(log_img, center_rc, max_radius_px=80,
                                 drop_factor=RADIUS_DROP_FACTOR):
    """
    Stima il raggio di un alone dal profilo radiale di log-densità.

    """
    r, c = int(center_rc[0]), int(center_rc[1])
    H, W = log_img.shape
    threshold = log_img[r, c] - np.log10(drop_factor)

    radii_found, n_nodrop = [], 0
    for angle in np.linspace(0.0, 2.0 * np.pi, 8, endpoint=False):
        sin_a, cos_a = np.sin(angle), np.cos(angle)
        found = None
        for d in range(1, max_radius_px):
            # P1-01: wrap periodico — nessuna distorsione al bordo
            rr = int(round(r + d * sin_a)) % H
            cc = int(round(c + d * cos_a)) % W
            if log_img[rr, cc] < threshold:
                found = float(d)
                break
        if found is not None:
            radii_found.append(found)
        else:
            n_nodrop += 1    # direzione senza caduta: esclusa dalla mediana

    if radii_found:
        return float(np.median(radii_found)), n_nodrop
    return DEFAULT_RADIUS_PX, n_nodrop    # fallback: TUTTE le direzioni senza caduta


def make_label_maps(img_size, centers_rc, radii_px,
                    sigma_factor=GAUSSIAN_SIGMA_FACTOR):
    """
    Genera heatmap gaussiana e radius map con geometria PERIODICA:
    - Distanze calcolate con metrica periodica min(|d|, img-|d|).
    - Pixel scritti con wrap modulo img_size (coerente con periodic=True).
    - Blob calcolato su finestra locale (O(half²), non O(img_size²)).

    """
    heatmap    = np.zeros((img_size, img_size), dtype=np.float32)
    radius_map = np.zeros((img_size, img_size), dtype=np.float32)

    for (rc, cc), rad in zip(centers_rc, radii_px):
        rc, cc, rad = int(rc), int(cc), float(rad)
        sigma  = max(1.0, rad * sigma_factor)
        r_draw = max(1.5, rad)
        # finestra locale: abbastanza grande da contenere il blob + il disco
        half   = int(np.ceil(max(4.0 * sigma, r_draw)))

        # coordinate NON wrappate della finestra
        ys = np.arange(rc - half, rc + half + 1)
        xs = np.arange(cc - half, cc + half + 1)
        rows = ys % img_size    # indici fisici con wrap periodico
        cols = xs % img_size

        # distanza periodica 2D
        YY, XX = np.meshgrid(ys, xs, indexing='ij')
        dy = np.abs(YY - rc); dy = np.minimum(dy, img_size - dy)
        dx = np.abs(XX - cc); dx = np.minimum(dx, img_size - dx)
        d2 = dx * dx + dy * dy

        # heatmap: blob gaussiano, merge con maximum (no saturazione)
        blob  = np.exp(-d2 / (2.0 * sigma * sigma)).astype(np.float32)
        sub_h = heatmap[np.ix_(rows, cols)]
        heatmap[np.ix_(rows, cols)] = np.maximum(sub_h, blob)

        # radius map: disco pieno, merge con maximum
        disk_mask = (d2 <= r_draw * r_draw)
        sub_r = radius_map[np.ix_(rows, cols)]
        radius_map[np.ix_(rows, cols)] = np.where(
            disk_mask, np.maximum(sub_r, np.float32(rad)), sub_r)

    return heatmap, radius_map

# %% ── 6c-1. De-duplicazione 3D — FIX tolleranza fissa ────────────────────────
# Fix: la tolleranza max(R_a,R_b) creava catene da 86-106 membri (raggi gonfiati
# dal criterio 1/e). Lo stesso oggetto 3D proiettato in fette adiacenti ha lo
# stesso centro xy → la scala giusta è la risoluzione del peak finder (~5 px).

def deduplicate_peaks_3d(peak_catalogue, lbox, tol_match=0.122, write_back=True):
    """
    Raggruppa i picchi dello stesso oggetto 3D in fette adiacenti.
    Match: distanza periodica xy < tol_match (FISSA, in Mpc/h).
    Aggiunge 'group_id' e 'is_primary' a ogni dict del catalogue.
    """
    xs, ys, rs, vs, sl = [], [], [], [], []
    for info in peak_catalogue:
        n = len(info["radii_px"])
        xs.append(info["peaks_mpc"][:, 0]); ys.append(info["peaks_mpc"][:, 1])
        rs.append(np.asarray(info["radii_mpc"],    dtype=float))
        vs.append(np.asarray(info["peak_logvals"], dtype=float))
        sl.append(np.full(n, info["slice_idx"], dtype=int))
    xs = np.concatenate(xs); ys = np.concatenate(ys)
    rs = np.concatenate(rs); vs = np.concatenate(vs); sl = np.concatenate(sl)
    N = len(xs)

    parent = np.arange(N)
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[max(ra, rb)] = min(ra, rb)

    def d2_periodic(dx, dy):
        dx = np.abs(dx); dx = np.minimum(dx, lbox - dx)
        dy = np.abs(dy); dy = np.minimum(dy, lbox - dy)
        return dx*dx + dy*dy

    slices_present = np.unique(sl)
    n_pairs = 0
    for s0, s1 in zip(slices_present[:-1], slices_present[1:]):
        if s1 - s0 != 1:
            continue
        ia, ib = np.where(sl == s0)[0], np.where(sl == s1)[0]
        if len(ia) == 0 or len(ib) == 0:
            continue
        D2   = d2_periodic(xs[ia][:, None] - xs[ib][None, :],
                           ys[ia][:, None] - ys[ib][None, :])
        cand = np.where(D2 < tol_match ** 2)
        for a, b in zip(*cand):
            union(ia[a], ib[b])
        n_pairs += len(cand[0])

    roots = np.array([find(i) for i in range(N)])
    _, group_id = np.unique(roots, return_inverse=True)
    is_primary = np.zeros(N, dtype=bool)
    for g in range(group_id.max() + 1):
        mem = np.where(group_id == g)[0]
        is_primary[mem[np.argmax(vs[mem])]] = True

    groups = []
    for g in range(group_id.max() + 1):
        mem = np.where(group_id == g)[0]
        p   = mem[np.argmax(vs[mem])]
        groups.append({
            "group_id": int(g), "n_members": int(len(mem)),
            "slices": sorted(set(sl[mem].tolist())),
            "x_prim": float(xs[p]), "y_prim": float(ys[p]),
            "R_max_mpc": float(rs[mem].max()),
            "logval_max": float(vs[mem].max()),
            "is_multi": bool(len(mem) > 1),
        })

    if write_back:
        i0 = 0
        for info in peak_catalogue:
            n = len(info["radii_px"])
            info["group_id"]   = group_id[i0:i0+n].copy()
            info["is_primary"] = is_primary[i0:i0+n].copy()
            i0 += n

    n_multi = sum(g["is_multi"] for g in groups)
    print(f"  tol={tol_match:.3f} Mpc/h ({tol_match/pixel_size_mpc:.1f} px) | "
          f"match={n_pairs} | gruppi={len(groups)} | multi={n_multi} | "
          f"secondari={(~is_primary).sum()} ({(~is_primary).sum()/N*100:.1f}%) | "
          f"max membri={max(g['n_members'] for g in groups)}")
    return groups


## 4) Rendering (o cache) delle slice e costruzione delle label
Costruzione delle fette 2D, label per fetta, cache raw e salvataggio del catalogo con de-duplicazione inter-fetta.


In [ ]:
# ── Loop principale: una fetta Z alla volta (con cache) ─────────────────────
# Se slices_density_raw.npy e slices_hmap_raw.npy esistono, il rendering SPH
# viene saltato e le fette caricate dalla cache (dipendono solo da snapshot +
# parametri SPH, non dalle soglie dei picchi). Per rigenerarle dall'inizio:
# cancella i due file .npy dalla cartella di output ed esegui di nuovo.
FORCE_RENDER = False

D_CACHE = os.path.join(OUTPUT_DIR, "slices_density_raw.npy")
H_CACHE = os.path.join(OUTPUT_DIR, "slices_hmap_raw.npy")
USE_CACHE = (not FORCE_RENDER) and os.path.exists(D_CACHE) and os.path.exists(H_CACHE)

if USE_CACHE:
    dens_cache = np.load(D_CACHE)
    hmap_cache = np.load(H_CACHE)
    assert len(dens_cache) == N_SLICES, "Cache incompatibile con N_SLICES attuale"
    print(f"Fast-path: carico {len(dens_cache)} fette dalla cache (rendering saltato).")
    print("  Per ri-renderizzare: FORCE_RENDER = True in testa a questa cella.\n")
else:
    print(f"Nessuna cache valida: rendering completo di {N_SLICES} fette.\n")

images_density_list  = []
images_hmap_list     = []
heatmaps_list        = []
radius_maps_list     = []
peak_catalogue       = []
slice_info           = []

for i in range(N_SLICES):
    z_lo = i * SLICE_DZ
    z_hi = z_lo + SLICE_DZ
    zc   = (z_lo + z_hi) / 2.0

    p_mask = (z >= z_lo) & (z < z_hi)
    n_sel  = p_mask.sum()
    if n_sel < 10:
        print(f"  Fetta {i:02d}: solo {n_sel} particelle — salto.")
        continue

    if USE_CACHE:
        # Fette già renderizzate: cast a float32 e via
        img_density = np.ascontiguousarray(dens_cache[i])
        h_map       = np.ascontiguousarray(hmap_cache[i])
    else:
        # ── Channel 1: rendering SPH density ────────────────────────────────
        img_density, _ = sphviewer2.render(
            x[p_mask], y[p_mask], z[p_mask],
            h_sph[p_mask], m[p_mask],
            Lbox=Lbox, extent=Lbox,
            xc=Lbox/2.0, yc=Lbox/2.0, zc=zc,
            azimuth=0.0, elevation=0.0, roll=0.0,
            periodic=True,
            r_max=R_MAX_LEVEL,
            target_cells_per_h=TARGET_CELLS,
            num_threads=os.cpu_count(),
        )
        # ── Channel 2: h_sph proiettato su griglia ──────────────────────────
        h_map = project_h_to_grid(x[p_mask], y[p_mask], h_sph[p_mask],
                                  lbox=Lbox, img_size=IMG_SIZE)
        img_density = img_density.astype(np.float32)   # P2-01
        h_map       = h_map.astype(np.float32)

    # ── Rilevamento picchi ──────────────────────────────────────────────────
    log_density = np.log10(img_density + EPS)
    peaks_rc, peak_vals, log_smooth = find_density_peaks(
        log_density,
        min_distance_px=PEAK_MIN_DISTANCE,
        threshold_sigma=PEAK_THRESHOLD_SIG,
    )

    # ── Stima raggi sull'immagine smoothata ─────────────────────────────────
    radii_px   = np.empty(len(peaks_rc), dtype=float)
    n_fallback = 0
    for j, pk in enumerate(peaks_rc):
        radii_px[j], n_nodrop = estimate_radius_from_profile(
            log_smooth, pk, drop_factor=RADIUS_DROP_FACTOR)
        if n_nodrop == 8:
            n_fallback += 1

    # ── Label periodiche ────────────────────────────────────────────────────
    heatmap, radius_map = make_label_maps(IMG_SIZE, peaks_rc, radii_px)

    # ── Accumula ────────────────────────────────────────────────────────────
    images_density_list.append(img_density)
    images_hmap_list.append(h_map)
    heatmaps_list.append(heatmap)
    radius_maps_list.append(radius_map)
    slice_info.append((z_lo, z_hi, len(peaks_rc)))

    peaks_mpc = (peaks_rc[:, ::-1] + 0.5) / IMG_SIZE * Lbox   # P1-03
    radii_mpc = radii_px / IMG_SIZE * Lbox
    peak_catalogue.append({
        "slice_idx": i, "z_lo": z_lo, "z_hi": z_hi,
        "peaks_px": peaks_rc.copy(), "peaks_mpc": peaks_mpc,
        "radii_px": radii_px.copy(), "radii_mpc": radii_mpc,
        "peak_logvals": peak_vals.copy(),
    })

    print(f"  Fetta {i:02d}  z=[{z_lo:.3f}, {z_hi:.3f}]  | {n_sel:>9,} part  "
          f"| {len(peaks_rc):>4} picchi  | {n_fallback} fallback")

n_slices_ok = len(images_density_list)
print(f"\nFette generate : {n_slices_ok}")
print(f"Picchi totali  : {sum(s[2] for s in slice_info)}")


# ── Salvataggio fette raw (solo se appena renderizzate) ──────────────────────
if not USE_CACHE:
    SLICES_D_PATH = os.path.join(OUTPUT_DIR, "slices_density_raw.npy")
    SLICES_H_PATH = os.path.join(OUTPUT_DIR, "slices_hmap_raw.npy")
    np.save(SLICES_D_PATH, np.stack(images_density_list).astype(np.float32))
    np.save(SLICES_H_PATH, np.stack(images_hmap_list).astype(np.float32))
    print(f"Fette raw salvate in:\n  {SLICES_D_PATH}")
else:
    print("Fette raw: già in cache, nessun salvataggio necessario.")


# ── De-duplicazione 3D definitiva + salvataggio catalogue ─────────────────────
TOL_MATCH = 0.122   # Mpc/h (~5 px: risoluzione del peak finder)

groups_summary = deduplicate_peaks_3d(peak_catalogue, Lbox,
                                      tol_match=TOL_MATCH, write_back=True)
np.save(os.path.join(OUTPUT_DIR, "peak_catalogue.npy"),
        np.array(peak_catalogue, dtype=object), allow_pickle=True)
print(f"Catalogo salvato (tolleranza {TOL_MATCH} Mpc/h).")


## 5) De-duplicazione e stime fisiche (massa/raggio)
Calcolo della massa d'apertura e confronto con il raggio stimato dal profilo 1/e.


In [ ]:
# ── Massa d'apertura dei picchi (verso la soglia fisica) ────────────────────
rho_c = 2.775e11   # densità critica z=0, (M☉/h)/(Mpc/h)³

# (1) Calibrazione unità per fetta: mpu × valore_img = M☉/h per pixel
mpu_list = []
for info in peak_catalogue:
    n_sl = int(((z >= info["z_lo"]) & (z < info["z_hi"])).sum())
    mpu_list.append((n_sl * dm_mass) / float(images_density_list[peak_catalogue.index(info)].sum()))
mpu_arr = np.array(mpu_list)
print(f"=== Calibrazione unità ===\n  mpu per fetta: min={mpu_arr.min():.4g}  max={mpu_arr.max():.4g}")
print("  (coerenti tra fette → unità render uniformi)")

# (2) M_ap(<R_1/e) con distanze periodiche, background sottratto
idx = 0
for pos, info in enumerate(peak_catalogue):
    img = images_density_list[pos]
    bg  = float(np.median(img))
    mpu = mpu_list[pos]
    pk  = np.asarray(info["peaks_px"]); rpx = np.asarray(info["radii_px"], float)
    m_ap = np.empty(len(pk))
    for k in range(len(pk)):
        rc, cc = int(pk[k,0]), int(pk[k,1])
        r = max(1.0, rpx[k]); ri = int(np.ceil(r))
        ys = np.arange(rc-ri, rc+ri+1) % IMG_SIZE
        xs = np.arange(cc-ri, cc+ri+1) % IMG_SIZE
        YY, XX = np.meshgrid(ys, xs, indexing='ij')
        dy = np.abs(YY-rc); dy = np.minimum(dy, IMG_SIZE-dy)
        dx = np.abs(XX-cc); dx = np.minimum(dx, IMG_SIZE-dx)
        disk = (dx*dx+dy*dy) <= r*r
        m_ap[k] = (img[np.ix_(ys,xs)][disk].sum() - bg*disk.sum()) * mpu
    info["M_ap_msun"]    = m_ap
    info["R200_est_mpc"] = (3.0*np.maximum(m_ap,1e-10)/(800*np.pi*rho_c))**(1/3)

all_M  = np.concatenate([i["M_ap_msun"]    for i in peak_catalogue])
all_R2 = np.concatenate([i["R200_est_mpc"] for i in peak_catalogue])
all_R1 = np.concatenate([np.asarray(i["radii_mpc"], float) for i in peak_catalogue])

print("\n=== Conteggi per soglia di massa ===")
for Mmin in [MASS_CUT_MSUN, 1e10, 3e10, 1e11, 3e11]:
    n = int((all_M >= Mmin).sum())
    print(f"  M_ap ≥ {Mmin:.1e} : {n:>6,}  ({n/len(all_M)*100:5.1f}%)")

print(f"\n=== R200 dalla massa vs criterio 1/e ===")
rat = all_R2 / np.maximum(all_R1, 1e-6)
print(f"  R200/R_1e : mediana={np.median(rat):.2f}  P16={np.percentile(rat,16):.2f}  P84={np.percentile(rat,84):.2f}")
print(f"  R200 max  : {all_R2.max()*1000:.0f} ckpc/h  (2R = {2*all_R2.max()*1000:.0f} vs Δz = {SLICE_DZ*1000:.0f})")

# (3) Plot: massa d'apertura vs raggio 1/e
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(all_R1*1000, np.log10(np.maximum(all_M, 1e6)), s=4, alpha=0.35,
           color='steelblue')
ax.axhline(np.log10(MASS_CUT_MSUN), color='lime', ls='--', lw=2,
           label=f'MASS_CUT = {MASS_CUT_MSUN:.1e}')
ax.set_xlabel('Raggio 1/e [ckpc/h]')
ax.set_ylabel(r'$\log_{10}(M_{ap}\ [M_\odot/h])$')
ax.set_title("Massa d'apertura vs raggio stimato")
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "mass_radius_plane.png"), dpi=150)
plt.show()


## 6) Normalizzazione
Normalizzazione robusta dei due canali su tutte le slice per coerenza inter-fetta.


In [ ]:
# ── Normalizzazione robusta ───────────────────────────────────────────────────
# CH1 (density): log10 + z-score robusto (mediana + sigma P84-P16)
# CH2 (h_map)  : -log10(h) per allineare il segno + stessa normalizzazione
# Entrambi calcolati su TUTTE le fette insieme per coerenza inter-fetta.

all_log_density = np.concatenate(
    [np.log10(img.ravel().astype(np.float64) + EPS) for img in images_density_list]
)
MED_D  = float(np.median(all_log_density))
SIG_D  = (np.percentile(all_log_density, 84) - np.percentile(all_log_density, 16)) / 2.0

all_neg_log_h = np.concatenate(
    [-np.log10(img.ravel().astype(np.float64) + EPS) for img in images_hmap_list]
)
MED_H  = float(np.median(all_neg_log_h))
SIG_H  = (np.percentile(all_neg_log_h, 84) - np.percentile(all_neg_log_h, 16)) / 2.0

print("=== Statistiche normalizzazione ===")
print(f"  CH1 density : mediana={MED_D:.4f}  sigma_rob={SIG_D:.4f}")
print(f"  CH2 h_map   : mediana={MED_H:.4f}  sigma_rob={SIG_H:.4f}")


def normalize_density(img):
    return (np.log10(img.astype(np.float64) + EPS) - MED_D) / SIG_D

def normalize_hmap(h_map):
    return (-np.log10(h_map.astype(np.float64) + EPS) - MED_H) / SIG_H


images_density_norm = [normalize_density(img) for img in images_density_list]
images_hmap_norm    = [normalize_hmap(img)    for img in images_hmap_list]

norm_stats = np.array([[MED_D, SIG_D], [MED_H, SIG_H]], dtype=np.float32)
np.save(os.path.join(OUTPUT_DIR, "norm_stats.npy"), norm_stats)
print(f"Norm stats salvate.")


## 7) Estrazione patch, salvataggio e controlli finali
Estrazione del dataset finale, salvataggio artefatti e riepilogo sintetico delle statistiche.


In [ ]:
# ── Estrazione patch ───────────────────────────────────────────────────────────
# X: (N, 2, PATCH_SIZE, PATCH_SIZE) float32
# Y_heatmap: (N, PATCH_SIZE, PATCH_SIZE) float32
# Y_radius:  (N, PATCH_SIZE, PATCH_SIZE) float32
# Tutte le patch incluse (anche quelle vuote): la rete impara che il cielo è perlopiù vuoto.

def extract_patches_2ch(density_norm, hmap_norm, heatmap, radius_map, patch_size):
    H, W = density_norm.shape
    x_patches, y_heat_patches, y_rad_patches = [], [], []
    for r in range(0, H - patch_size + 1, patch_size):
        for c in range(0, W - patch_size + 1, patch_size):
            ch1 = density_norm[r:r+patch_size, c:c+patch_size].astype(np.float32)
            ch2 = hmap_norm   [r:r+patch_size, c:c+patch_size].astype(np.float32)
            x_patches     .append(np.stack([ch1, ch2], axis=0))
            y_heat_patches.append(heatmap   [r:r+patch_size, c:c+patch_size])
            y_rad_patches .append(radius_map[r:r+patch_size, c:c+patch_size])
    return x_patches, y_heat_patches, y_rad_patches


X_patches, Y_heat_patches, Y_rad_patches = [], [], []
for dn, hn, heat, rmap in zip(images_density_norm, images_hmap_norm,
                               heatmaps_list, radius_maps_list):
    xp, yh, yr = extract_patches_2ch(dn, hn, heat, rmap, PATCH_SIZE)
    X_patches      .extend(xp)
    Y_heat_patches .extend(yh)
    Y_rad_patches  .extend(yr)

X      = np.stack(X_patches)       .astype(np.float32)
Y_heat = np.stack(Y_heat_patches)  .astype(np.float32)
Y_rad  = np.stack(Y_rad_patches)   .astype(np.float32)

n_pos = (Y_heat.max(axis=(1, 2)) > 0.05).sum()

print("=== Patch dataset ===")
print(f"  X shape          : {X.shape}  (N, canali, H, W)")
print(f"  Y_heatmap shape  : {Y_heat.shape}")
print(f"  Y_radius shape   : {Y_rad.shape}")
print(f"  Patch positive   : {n_pos} / {len(X)}  ({n_pos/len(X)*100:.1f}%)")
print(f"  Max heatmap      : {Y_heat.max():.4f}  (deve essere ≈ 1.0)")
print(f"  Max radius (px)  : {Y_rad.max():.1f}")
print(f"  Memoria X        : {X.nbytes/1e6:.1f} MB")


# Conteggio picchi totale per i controlli finali
all_radii = np.concatenate([info["radii_px"] for info in peak_catalogue if len(info["radii_px"]) > 0]) \
            if any(len(info["radii_px"]) > 0 for info in peak_catalogue) else np.array([])

# ── Salvataggio dataset ────────────────────────────────────────────────────────

X_path   = os.path.join(OUTPUT_DIR, "X_patches.npy")
Yh_path  = os.path.join(OUTPUT_DIR, "Y_heatmap_patches.npy")
Yr_path  = os.path.join(OUTPUT_DIR, "Y_radius_patches.npy")
cat_path = os.path.join(OUTPUT_DIR, "peak_catalogue.npy")

np.save(X_path,   X)
np.save(Yh_path,  Y_heat)
np.save(Yr_path,  Y_rad)
np.save(cat_path, np.array(peak_catalogue, dtype=object), allow_pickle=True)

print("=== Dataset salvato ===")
print(f"  X          → {X_path}      ({X.nbytes/1e6:.1f} MB)")
print(f"  Y_heatmap  → {Yh_path}  ({Y_heat.nbytes/1e6:.1f} MB)")
print(f"  Y_radius   → {Yr_path}   ({Y_rad.nbytes/1e6:.1f} MB)")
print(f"  Catalogo   → {cat_path}")

info_lines = [
    "=== CAMELS Preprocessing v2.1 — Heatmap Labels from SPH ===",
    f"Snapshot        : snapshot_090.hdf5  (z = {redshift:.4f})",
    f"Box size        : {Lbox:.4f} Mpc/h",
    f"N particles     : {len(x):,}  ({round(len(x)**(1/3))}³)",
    f"DM particle mass: {dm_mass:.3e} M☉/h",
    "",
    "--- Input (2 canali) ---",
    f"CH1 density     : SPH render  r_max={R_MAX_LEVEL}  target_cells={TARGET_CELLS}",
    f"CH2 h_map       : proiezione media h_sph su griglia  k={N_NEIGHBORS_H}  "
    f"+ gaussian_filter(σ=1, grid-wrap)  [P2-02]",
    "",
    "--- Label ---",
    "Tipo            : Gaussian heatmap (CenterNet-style) + radius map, periodiche  [P0-01]",
    f"Rilevamento     : max-filter grid-wrap  min_dist={PEAK_MIN_DISTANCE}px  "
    f"thresh={PEAK_THRESHOLD_SIG}σ  [P0-03]",
    f"Radius crit.    : su immagine smoothata, wrap periodico  [P0-02, P1-01]",
    f"Gaussian σ      : R_px × {GAUSSIAN_SIGMA_FACTOR}",
    "",
    "--- Slicing ---",
    f"N slices        : {N_SLICES}",
    f"Slice Δz        : {SLICE_DZ*1000:.1f} ckpc/h = {SLICE_DZ:.4f} Mpc/h",
    "",
    "--- Normalizzazione ---",
    f"CH1 MED/SIG     : {MED_D:.4f} / {SIG_D:.4f}  (log10 + robust z-score)",
    f"CH2 MED/SIG     : {MED_H:.4f} / {SIG_H:.4f}  (-log10(h) + robust z-score)",
    "",
    "--- Patch dataset ---",
    f"Patch size      : {PATCH_SIZE} × {PATCH_SIZE} px",
    f"Total patches   : {len(X):,}",
    f"Positive patches: {n_pos:,}  ({n_pos/len(X)*100:.1f}%)",
    f"X shape         : {X.shape}",
    f"Y_heatmap shape : {Y_heat.shape}",
    f"Y_radius shape  : {Y_rad.shape}",
    f"Picchi totali   : {len(all_radii)}",
    "",
    "--- NOTA training (P2-04) ---",
    "Normalizzare Y_rad prima del training per stabilità della regressione L1:",
    "  opzione A) Y_rad_norm = Y_rad / 80.0          (lineare, max atteso ≈ 80 px)",
    "  opzione B) Y_rad_norm = log1p(Y_rad) / log1p(80)  (compressa, robusta agli outlier)",
    "",
    "--- File di output ---",
    "X_patches.npy, Y_heatmap_patches.npy, Y_radius_patches.npy",
    "norm_stats.npy, peak_catalogue.npy",
    "slices_density_raw.npy, slices_hmap_raw.npy  [P1-04: per calibrazione soglia]",
    "smoothing_lengths_090_k32.npy",
]
info_text = "\n".join(info_lines)
print("\n" + info_text)
with open(os.path.join(OUTPUT_DIR, "dataset_info.txt"), "w", encoding="utf-8") as f:
    f.write(info_text)


# ── Aggiornamento dataset_info con blocco de-duplicazione ─────────────────────
info_path = os.path.join(OUTPUT_DIR, "dataset_info.txt")
with open(info_path, "r", encoding="utf-8") as f:
    txt = f.read()

n_primtot = sum(len(i["radii_px"]) for i in peak_catalogue)
n_sec   = sum(int((~np.asarray(i["is_primary"], dtype=bool)).sum())
              for i in peak_catalogue)

txt += (
    "\n--- De-duplicazione inter-fetta (follow-up P1-05, FIX tolleranza fissa) ---\n"
    f"Picchi totali       : {n_primtot:,}\n"
    f"Oggetti unici (3D)  : {len(groups_summary)}\n"
    f"Gruppi multi-fetta  : {sum(g['is_multi'] for g in groups_summary)}"
    f"   (picchi secondari: {n_sec}, {n_sec/max(n_primtot,1)*100:.1f}%)\n"
    f"Tolleranza match    : {TOL_MATCH:.3f} Mpc/h ({TOL_MATCH/pixel_size_mpc:.0f} px), FISSA in xy\n"
    "Scelta (opzione A)  : label invariate per tutte le fette; le proiezioni dello\n"
    "  stesso oggetto 3D in fette adiacenti sono fisicamente diverse e legittime.\n"
    "  'is_primary' va usato SOLO in validazione (es. HMF: contare i soli primari).\n"
    "Nota fisica         : raggi > 500 ckpc/h sovrastimati dal criterio 1/e sul\n"
    "  campo proiettato; le masse implicite sono limiti superiori.\n"
)
with open(info_path, "w", encoding="utf-8") as f:
    f.write(txt)
print("dataset_info.txt aggiornato con il blocco de-duplicazione.")
